TESTE DE IMPORTAÇÃO DLT

In [16]:
import dlt
import numpy

In [14]:
data = [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]

pipeline = dlt.pipeline(
    pipeline_name="quick_start", destination="duckdb", dataset_name="mydata"
)
load_info = pipeline.run(data, table_name="users")

print(load_info)

Pipeline quick_start load step finished in 0.23 seconds
1 load package(s) were loaded to destination duckdb and into dataset mydata
The duckdb destination used duckdb:///c%3A%5CUsers%5Cpedro.lourega%5CDocuments%5CGitHub%5Cstudy%5Cdlt_study%5Cdlt_postgres_to_postgres%5Cnotebooks%5Cquick_start.duckdb location to store data
Load package 1794842842.4164677 is LOADED and contains no failed jobs


In [19]:
dataset = pipeline.dataset()

table = dataset.users

print(table.df())
print(table.limit(10).arrow())

   id   name        _dlt_load_id         _dlt_id
0   1  Alice  1794841941.2408261  Okiwiz4mm5frag
1   2    Bob  1794841941.2408261  MmWY6XIw/Gf+UQ
2   1  Alice  1794842842.4164677  meTRAEKH6FoyZA
3   2    Bob  1794842842.4164677  bxVlcBBLroLtMA
pyarrow.Table
id: int64
name: string
_dlt_load_id: string
_dlt_id: string
----
id: [[1,2,1,2]]
name: [["Alice","Bob","Alice","Bob"]]
_dlt_load_id: [["1794841941.2408261","1794841941.2408261","1794842842.4164677","1794842842.4164677"]]
_dlt_id: [["Okiwiz4mm5frag","MmWY6XIw/Gf+UQ","meTRAEKH6FoyZA","bxVlcBBLroLtMA"]]


In [ ]:
import dlt
from dlt.sources.helpers import requests

url = "https://api.github.com/repos/dlt-hub/dlt/issues"

response = requests.get(url)

response.raise_for_status()

pipeline = dlt.pipeline(
    pipeline_name="github_issues_pipeline_00",
    destination="duckdb",
    dataset_name="github_issues",
)

pipeline.run(response.json(), table_name="issues")

2026-11-16 13:45:07,856|[WARNING]|16092|16476|dlt|validate.py|verify_normalized_table:113|In schema `github_issues`: The following columns in table 'issues' did not receive any data during this load and therefore could not have their types inferred:
  - active_lock_reason
  - assignee
  - closed_at
  - closed_by
  - milestone
  - performed_via_github_app
  - pinned_comment
  - pull_request__merged_at
  - state_reason
  - type

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'active_lock_reason': {'data_type': 'text'}})

2026-11-16 13:45:07,857|[WARNING]|16092|16476|dlt|validate.py|verify_normalized_table:113|In schema `github_issues`: The following columns in table 'issues__labels' did not receive any data during this load and therefore could not have their types inferred:
  - archived_at
  - archived_by

Unles

LoadInfo(pipeline=<dlt.pipeline(pipeline_name='github_issues_pipeline', destination='duckdb', dataset_name='github_issues', default_schema_name='github_issues', schema_names=['github_issues'], first_run=False, dev_mode=False, is_active=True, pipelines_dir='C:\\Users\\pedro.lourega\\.dlt\\pipelines', working_dir='C:\\Users\\pedro.lourega\\.dlt\\pipelines\\github_issues_pipeline')>, metrics={'1794847507.636402': [{'started_at': DateTime(2026, 11, 16, 16, 45, 7, 960200, tzinfo=Timezone('UTC')), 'finished_at': DateTime(2026, 11, 16, 16, 45, 8, 613245, tzinfo=Timezone('UTC')), 'job_metrics': {'issues.f2db8f6d34.insert_values.gz': LoadJobMetrics(job_id='issues.f2db8f6d34.insert_values.gz', file_path='C:\\Users\\pedro.lourega\\.dlt\\pipelines\\github_issues_pipeline\\load\\normalized\\1794847507.636402\\started_jobs\\issues.f2db8f6d34.0.insert_values.gz', table_name='issues', started_at=datetime.datetime(2026, 11, 16, 16, 45, 8, 120103, tzinfo=datetime.timezone.utc), finished_at=datetime.date

In [24]:
from os import write


url = "https://api.github.com/repos/dlt-hub/dlt/issues"

response = requests.get(url)

response.raise_for_status()

pipeline = dlt.pipeline(
    pipeline_name="github_issues_pipeline_01",
    destination="duckdb",
    dataset_name="github_data",
)

load_info = pipeline.run(
    response.json(), table_name="issues", write_disposition="replace"
)

print(load_info)

2026-11-16 13:50:32,336|[WARNING]|16092|16476|dlt|validate.py|verify_normalized_table:113|In schema `github_issues_pipeline_01`: The following columns in table 'issues' did not receive any data during this load and therefore could not have their types inferred:
  - active_lock_reason
  - assignee
  - closed_at
  - closed_by
  - milestone
  - performed_via_github_app
  - pinned_comment
  - pull_request__merged_at
  - state_reason
  - type

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'active_lock_reason': {'data_type': 'text'}})

2026-11-16 13:50:32,338|[WARNING]|16092|16476|dlt|validate.py|verify_normalized_table:113|In schema `github_issues_pipeline_01`: The following columns in table 'issues__labels' did not receive any data during this load and therefore could not have their types inferred:
  - archived_a

Pipeline github_issues_pipeline_01 load step finished in 0.61 seconds
1 load package(s) were loaded to destination duckdb and into dataset github_data
The duckdb destination used duckdb:///c%3A%5CUsers%5Cpedro.lourega%5CDocuments%5CGitHub%5Cstudy%5Cdlt_study%5Cdlt_postgres_to_postgres%5Cnotebooks%5Cgithub_issues_pipeline_01.duckdb location to store data
Load package 1794847832.075599 is LOADED and contains no failed jobs


INSERÇÃO DE DADOS AOS POUCOS / CARREGAMENTO INCREMENTAL

In [26]:
from typing import Any, Iterator

import dlt
from dlt.sources.helpers import requests


@dlt.resource(table_name="issues", write_disposition="append")
def get_issues(
    created_at: dlt.sources.incremental[str] = dlt.sources.incremental(
        "created_at", initial_value="1970-01-01T00:00:00Z"
    ),
) -> Iterator[dict[str, Any]]:
    url = (
        "https://api.github.com/repos/dlt-hub/dlt/issues"
        "?per_page=100&sort=created&directions=desc&state=open"
    )

    while True:
        response = requests.get(url)
        response.raise_for_status()
        yield response.json()

        if created_at.start_out_of_range:
            break

        if "next" not in response.links:
            break
        url = response.links["next"]["url"]


pipeline = dlt.pipeline(
    pipeline_name="github_issues_incremental",
    destination="duckdb",
    dataset_name="github_data_append",
)

load_info = pipeline.run(get_issues)
normalize_info = pipeline.last_trace.last_normalize_info


print(normalize_info.row_counts)

2026-11-16 14:09:24,583|[WARNING]|16092|16476|dlt|validate.py|verify_normalized_table:113|In schema `github_issues_incremental`: The following columns in table 'issues' did not receive any data during this load and therefore could not have their types inferred:
  - active_lock_reason
  - assignee
  - closed_at
  - closed_by
  - milestone
  - milestone__due_on
  - performed_via_github_app
  - pinned_comment
  - pull_request__merged_at
  - type

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'active_lock_reason': {'data_type': 'text'}})

2026-11-16 14:09:24,583|[WARNING]|16092|16476|dlt|validate.py|verify_normalized_table:113|In schema `github_issues_incremental`: The following columns in table 'issues__labels' did not receive any data during this load and therefore could not have their types inferred:
  - archi

{'issues': 493, 'issues__labels': 412, 'issues__performed_via_github_app__events': 108, 'issues__assignees': 162, '_dlt_pipeline_state': 1}
